# 03 – Model comparison

Comparing learned models with the baselines from `02_baseline.ipynb`, using the same split, preprocessing and cross-validation. The test set stays closed in this notebook and is only used once, on the final model.

## 1. Setup

In [1]:
import sys

import matplotlib.pyplot as plt
import pandas as pd
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import make_scorer, fbeta_score, precision_score, f1_score
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder

SEED = 42
TEST_SIZE = 0.2
N_FOLDS = 5

print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__)
print("scikit-learn:", sklearn.__version__)

Python: 3.13.11
pandas: 3.0.6
scikit-learn: 1.9.1


In [2]:
df = pd.read_csv("../data/ai4i2020.csv")

target = "Machine failure"
drop_cols = ["UDI", "Product ID", "TWF", "HDF", "PWF", "OSF", "RNF"]

X = df.drop(columns=drop_cols + [target])
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=SEED
)

print("Train:", X_train.shape, "Test:", X_test.shape)
print("Failures in train:", y_train.sum(), "Failures in test:", y_test.sum())

Train: (8000, 6) Test: (2000, 6)
Failures in train: 271 Failures in test: 68


In [3]:
numeric_cols = ["Air temperature [K]", "Process temperature [K]",
                "Rotational speed [rpm]", "Torque [Nm]", "Tool wear [min]"]
categorical_cols = ["Type"]

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), numeric_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
])

cv = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

scoring = {
    "f2": make_scorer(fbeta_score, beta=2, zero_division=0),
    "average_precision": "average_precision",
    "precision": make_scorer(precision_score, zero_division=0),
    "recall": "recall",
    "f1": make_scorer(f1_score, zero_division=0),
    "accuracy": "accuracy",
}

In [4]:
def evaluate(name, model):
    """Score a model with the shared pipeline and cross-validation folds."""
    pipe = Pipeline([("prep", preprocessor), ("model", model)])
    scores = cross_validate(pipe, X_train, y_train, cv=cv, scoring=scoring)
    row = {"model": name}
    for metric in scoring:
        fold_scores = scores[f"test_{metric}"]
        row[metric] = f"{fold_scores.mean():.3f} ± {fold_scores.std():.3f}"
    return row


rows = [
    evaluate("Dummy (most frequent)", DummyClassifier(strategy="most_frequent")),
    evaluate("Logistic Regression", LogisticRegression(max_iter=1000)),
]

results_table = pd.DataFrame(rows).set_index("model")
results_table

,f2,average_precision,precision,recall,f1,accuracy
model,,,,,,
Dummy (most frequent),0.000 ± 0.000,0.034 ± 0.000,0.000 ± 0.000,0.000 ± 0.000,0.000 ± 0.000,0.966 ± 0.000
Logistic Regression,0.233 ± 0.028,0.469 ± 0.032,0.761 ± 0.109,0.199 ± 0.028,0.312 ± 0.026,0.970 ± 0.001


**Check:** the split and baseline scores match `02_baseline.ipynb`, so the new models are compared under the same evaluation protocol. Every model is scored on the same folds of the training set, with preprocessing fitted inside each fold to avoid leakage.

## 2. k-Nearest Neighbours

kNN predicts from the most similar training rows, so it can capture local patterns that Logistic Regression's linear boundary misses. It depends on distances, so it uses the same scaling pipeline. I tune the number of neighbours and the weighting on the training folds, using F2 because it is the primary metric.

In [5]:
from sklearn.model_selection import GridSearchCV
from sklearn.neighbors import KNeighborsClassifier

knn_pipe = Pipeline([("prep", preprocessor), ("model", KNeighborsClassifier())])
knn_grid = {
    "model__n_neighbors": [1, 3, 5, 7, 11, 15, 21, 31],
    "model__weights": ["uniform", "distance"],
}

knn_search = GridSearchCV(knn_pipe, knn_grid, cv=cv, scoring=scoring["f2"], refit=False)
knn_search.fit(X_train, y_train)

print("Best settings:", knn_search.best_params_)
print("Best mean F2:", round(knn_search.best_score_, 3))

knn_tuning = pd.DataFrame(knn_search.cv_results_).pivot(
    index="param_model__n_neighbors",
    columns="param_model__weights",
    values="mean_test_score",
).round(3)
knn_tuning

Best settings: {'model__n_neighbors': 1, 'model__weights': 'uniform'}
Best mean F2: 0.469


param_model__weights,distance,uniform
param_model__n_neighbors,,
1,0.469,0.469
3,0.416,0.403
5,0.383,0.360
7,0.358,0.303
11,0.272,0.227
15,0.235,0.176
21,0.211,0.138
31,0.130,0.077


In [6]:
best_knn = KNeighborsClassifier(
    n_neighbors=knn_search.best_params_["model__n_neighbors"],
    weights=knn_search.best_params_["model__weights"],
)
knn_row = evaluate("kNN (tuned)", best_knn)

results_table = pd.DataFrame(rows + [knn_row]).set_index("model")
results_table

,f2,average_precision,precision,recall,f1,accuracy
model,,,,,,
Dummy (most frequent),0.000 ± 0.000,0.034 ± 0.000,0.000 ± 0.000,0.000 ± 0.000,0.000 ± 0.000,0.966 ± 0.000
Logistic Regression,0.233 ± 0.028,0.469 ± 0.032,0.761 ± 0.109,0.199 ± 0.028,0.312 ± 0.026,0.970 ± 0.001
kNN (tuned),0.469 ± 0.080,0.269 ± 0.067,0.543 ± 0.065,0.454 ± 0.085,0.493 ± 0.073,0.969 ± 0.004


**Findings:**
- The best F2 was at k = 1 (0.469 ± 0.080), compared with 0.233 ± 0.028 for Logistic Regression. Recall rose from 0.199 to 0.454, but precision fell from 0.761 to 0.543.
- F2 falls steadily as k grows. Failures are only 3.4% of the data, so a larger group of neighbours rarely has a failure majority at the default threshold.
- k = 1 has the most variable F2 of the models so far, and it can be sensitive to noisy labels, including the 18 inconsistent RNF rows.
- Average precision is lower than Logistic Regression's (0.269 vs 0.469). With k = 1, each prediction is only 0 or 1, so kNN has little ability to rank machines by risk.
- The same folds were used for tuning and reporting, so this score may be slightly optimistic.

## 3. Decision Tree

A decision tree splits the data on thresholds, so it can capture non-linear patterns and interactions between variables that Logistic Regression's linear boundary cannot. It does not need scaling, but it uses the same pipeline so every model is compared under identical conditions. I tune the depth, the minimum leaf size and class weighting on the training folds, using F2 as the selection metric.

In [7]:
from sklearn.tree import DecisionTreeClassifier

tree_pipe = Pipeline([("prep", preprocessor), ("model", DecisionTreeClassifier(random_state=SEED))])
tree_grid = {
    "model__max_depth": [3, 5, 7, 10, None],
    "model__min_samples_leaf": [1, 5, 10, 20],
    "model__class_weight": [None, "balanced"],
}

tree_search = GridSearchCV(tree_pipe, tree_grid, cv=cv, scoring=scoring["f2"], refit=False)
tree_search.fit(X_train, y_train)

print("Best settings:", tree_search.best_params_)
print("Best mean F2:", round(tree_search.best_score_, 3))

tree_tuning = (
    pd.DataFrame(tree_search.cv_results_)
    .sort_values("rank_test_score")
    [["param_model__max_depth", "param_model__min_samples_leaf",
      "param_model__class_weight", "mean_test_score", "std_test_score"]]
    .head(5)
    .round(3)
)
tree_tuning

Best settings: {'model__class_weight': None, 'model__max_depth': 10, 'model__min_samples_leaf': 5}
Best mean F2: 0.692


,param_model__max_depth,param_model__min_samples_leaf,param_model__class_weight,mean_test_score,std_test_score
13,10,5,NaN,0.692,0.049
17,None,5,NaN,0.691,0.050
37,None,5,balanced,0.690,0.045
16,None,1,NaN,0.677,0.051
12,10,1,NaN,0.675,0.063


In [8]:
best_tree = DecisionTreeClassifier(
    max_depth=tree_search.best_params_["model__max_depth"],
    min_samples_leaf=tree_search.best_params_["model__min_samples_leaf"],
    class_weight=tree_search.best_params_["model__class_weight"],
    random_state=SEED,
)
tree_row = evaluate("Decision Tree (tuned)", best_tree)

results_table = pd.DataFrame(rows + [knn_row, tree_row]).set_index("model")
results_table

,f2,average_precision,precision,recall,f1,accuracy
model,,,,,,
Dummy (most frequent),0.000 ± 0.000,0.034 ± 0.000,0.000 ± 0.000,0.000 ± 0.000,0.000 ± 0.000,0.966 ± 0.000
Logistic Regression,0.233 ± 0.028,0.469 ± 0.032,0.761 ± 0.109,0.199 ± 0.028,0.312 ± 0.026,0.970 ± 0.001
kNN (tuned),0.469 ± 0.080,0.269 ± 0.067,0.543 ± 0.065,0.454 ± 0.085,0.493 ± 0.073,0.969 ± 0.004
Decision Tree (tuned),0.692 ± 0.049,0.702 ± 0.044,0.793 ± 0.079,0.672 ± 0.048,0.726 ± 0.055,0.983 ± 0.004


**Findings:**
- The tree has the best F2 so far (0.692 ± 0.049), compared with 0.469 for kNN and 0.233 for Logistic Regression. It also has the best average precision (0.702), so it ranks risk better as well as classifying better at the default threshold.
- Precision (0.793) and recall (0.672) are both reasonably high, so the tree improves recall without losing precision, unlike kNN.
- A tree can split on thresholds and combine conditions, which suits failures that happen only when several conditions occur together. Logistic Regression can only weight each variable separately, which is a plausible reason for the large gap.
- The top three settings score 0.692, 0.691 and 0.690, all well within the fold variation of ±0.05. The exact settings matter much less than the model family, so I will not read much into the precise choice of depth 10.
- Class weighting did not help the best setting: the best unweighted score (0.692) is almost identical to the best balanced one (0.690).
- The same folds were used for tuning and reporting, so these scores may be slightly optimistic.